# Session 16 · Homework — SOLUTIONS (teacher)

Worked solutions with commentary. All cells run top to bottom.
**Key point:** the exact peak depth varies with the split seed — grade the *reasoning*
(held-out justification, detection, a second example), not a specific number.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../../datasets/anchor/student_habits.csv")
habits = ["study_hours_per_week","attendance_pct","sleep_hours_per_night","screen_time_hours_per_day","practice_sessions_per_week"]
X, y = df[habits], df['passed']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print('train students:', len(y_train), '  test students:', len(y_test))

## Part 1 · Twin-curve plot — SOLUTION

Training climbs monotonically to **1.000**; test peaks near depth **5–6 (~0.889)** then
falls to **~0.82** at unlimited depth. The gap widens from ~0.02 to ~0.18 = overfitting.

In [ ]:
depths = list(range(1, 16)) + [None]        # 1..15, then unlimited
rows = []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    rows.append({'max_depth': (99 if d is None else d),   # 99 = 'unlimited' on the x-axis
                 'label': ('none' if d is None else str(d)),
                 'train_acc': t.score(X_train, y_train),
                 'test_acc':  t.score(X_test,  y_test)})
curve = pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(curve['max_depth'], curve['train_acc'], 'o-', color='tab:red', label='training')
ax.plot(curve['max_depth'], curve['test_acc'], 'o-', color='tab:blue', label='test')
peak = curve.loc[curve['test_acc'].idxmax()]
ax.axvline(peak['max_depth'], ls='--', color='gray')
ax.set_xlabel('max_depth (99=unlimited)'); ax.set_ylabel('accuracy')
ax.set_title('Overfitting curve'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(curve[['label','train_acc','test_acc']].round(3).to_string(index=False))

## Part 2 · Unlimited tree — SOLUTION

Training **1.000**, test **~0.82**. The **test** accuracy is the honest measure: it
reports performance on students the model never saw, which is what we actually care
about. Training 1.000 just means the tree memorised its training set — it is the
*symptom* of overfitting, not a sign of quality.

In [ ]:
full_tree = DecisionTreeClassifier(max_depth=None, random_state=0).fit(X_train, y_train)
print('training accuracy:', round(full_tree.score(X_train, y_train), 3), '(memorising symptom)')
print('test accuracy    :', round(full_tree.score(X_test, y_test), 3), '(the honest measure)')

## Part 3 · Depth to ship — SOLUTION

Ship the depth at the **peak of the test curve** (~5–6 here). It generalises best to new
students. The deepest tree is perfect on training but worse on the students who actually
matter — the ones it hasn't seen. We tune capacity against **held-out** performance,
never against training.

In [ ]:
peak = curve.loc[curve['test_acc'].idxmax()]
print('ship max_depth =', peak['label'], ' test acc', round(peak['test_acc'],3))
print('deepest tree   =', curve.iloc[-1]['label'],
      ' test acc', round(curve.iloc[-1]['test_acc'],3), '(worse on new students)')

## Part 4 · Overfitting explanation — SOLUTION (sample)

> *Overfitting is when a model stops **learning** the general pattern and starts
> **memorising** the training data's accidents — building private rules for individual
> training points, including noise. It looks brilliant on the data it trained on (a deep
> tree hits 100% training accuracy) but does worse on new data, because those private
> rules don't generalise. You **detect** it by measuring accuracy on a **held-out test
> set**: if test accuracy is much lower than training accuracy — or starts falling as
> you add complexity — you're overfitting. The same danger appears with **KNN's `k`**:
> k=1 (Session 9) scored 100% on training by letting the single nearest point decide,
> exactly like an unlimited tree.*

**Grading:** require all three elements — the memorising-vs-learning framing, a
**detection method** (held-out data), and a **second example** (k=1 KNN, or over-
segmentation in clustering to come in S21). The specific peak depth is not required and
varies by seed; a sound held-out justification is what earns Part 3.